# Rolling MPS vs Classical Indicators — Single-Sector Banks (2005-2024, incl. 2008 GFC)
**MFE Industry Project · AQS**

MPS-derived indicators (fit on rolling 252-day windows, fixed Fiedler order) compared head-to-head with the
standard risk indicators, on the 12-asset **single-sector** US bank portfolio (JPM, BAC, WFC, C, GS, MS, USB, PNC, FITB, KEY, RF, HBAN) — a homogeneous, highly-correlated book (avg pairwise corr ~0.7), the contrast case to the diversified cross-asset book.

Classical set: **realized volatility, average pairwise correlation, first-PC explained variance (absorption
ratio), VIX, MOVE, drawdown, realized VaR breaches, crisis periods**. Questions: do the MPS indicators track
the classical ones (validation), flag crises, and carry **non-redundant** signal beyond vol/correlation/absorption?

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, sys
here = os.getcwd(); ROOT = here if os.path.isdir(os.path.join(here,"src")) else os.path.dirname(here)
def P(rel): return os.path.join(ROOT, rel)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from numpy.linalg import lstsq
CRISES = [('2008-09-01', '2009-03-31', 'GFC-2008'), ('2011-07-01', '2011-10-31', 'euro'), ('2015-08-01', '2016-02-29', 'China/oil'), ('2018-10-01', '2018-12-31', 'Q4-2018'), ('2020-02-15', '2020-04-30', 'COVID'), ('2022-01-01', '2022-10-31', 'rate-hikes'), ('2023-03-01', '2023-03-31', 'SVB')]
df = pd.read_csv(P("data/rolling_banks.csv"), index_col=0, parse_dates=True)
mps_cols = [c for c in df.columns if c.startswith("mps_")]
cls_cols = ["realized_vol","avg_corr","pc1_share","VIX","MOVE","drawdown","var_breaches"]
print("rolling table:", df.shape, "|", df.index.min().date(), "->", df.index.max().date())
print(df[["mps_entanglement","mps_total_corr","mps_tail_state","realized_vol","avg_corr","pc1_share","VIX"]].describe().round(3).to_string())

rolling table: (228, 14) | 2006-01-03 -> 2024-12-11
       mps_entanglement  mps_total_corr  mps_tail_state  realized_vol  avg_corr  pc1_share      VIX
count           228.000         228.000         228.000       228.000   228.000    228.000  228.000
mean              2.044          10.036           0.002         0.317     0.737      0.760   19.378
std               0.011           1.229           0.002         0.220     0.089      0.081    6.404
min               1.993           7.607           0.000         0.114     0.510      0.555   11.043
25%               2.038           9.182           0.000         0.191     0.677      0.705   14.564
50%               2.046          10.006           0.001         0.253     0.748      0.770   17.171
75%               2.052          10.758           0.002         0.328     0.799      0.817   23.111
max               2.067          13.109           0.018         1.168     0.898      0.907   41.014


## 1 · Overlay — do the MPS indicators move with market stress? (crisis windows shaded)

In [2]:
def z(s): return (s - s.mean())/s.std()
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
panels = [("MPS entanglement vs VIX & absorption (PC1)", ["mps_entanglement","VIX","pc1_share"]),
          ("MPS total-correlation & mean-MI vs avg-corr & MOVE", ["mps_total_corr","mps_mean_MI","avg_corr","MOVE"]),
          ("MPS tail-state prob & tail-asym vs drawdown & VaR breaches", ["mps_tail_state","mps_tail_asym","drawdown","var_breaches"])]
for ax,(title,keys) in zip(axes, panels):
    for k in keys: ax.plot(df.index, z(df[k]), label=k, lw=1.5)
    for s,e,lab in CRISES:
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color="red", alpha=0.10)
        ax.text(pd.Timestamp(s), ax.get_ylim()[1]*0.9, lab, fontsize=7, color="darkred")
    ax.set_title(title); ax.legend(fontsize=8, ncol=4); ax.axhline(0, color="k", lw=0.4)
axes[-1].set_xlabel("window-end date"); plt.tight_layout(); plt.savefig(P("data/rv_overlay.png"), dpi=110); plt.show()

## 2 · Spearman correlation — MPS features vs classical indicators
High |ρ| ⇒ the MPS feature re-expresses a classical indicator (redundant, but validates it). Uniformly low ⇒
it measures something else.

In [3]:
M = pd.DataFrame(index=mps_cols, columns=cls_cols, dtype=float)
for a in mps_cols:
    for b in cls_cols: M.loc[a,b] = spearmanr(df[a], df[b]).correlation
fig, ax = plt.subplots(figsize=(10,5.5)); im = ax.imshow(M.values.astype(float), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cls_cols))); ax.set_xticklabels(cls_cols, rotation=45, ha="right")
ax.set_yticks(range(len(mps_cols))); ax.set_yticklabels(mps_cols)
for i in range(len(mps_cols)):
    for j in range(len(cls_cols)): ax.text(j,i,"%.2f"%M.values[i,j], ha="center", va="center", fontsize=7)
plt.colorbar(im, fraction=0.046); ax.set_title("Spearman: MPS features (rows) vs classical (cols)")
plt.tight_layout(); plt.savefig(P("data/rv_corrtable.png"), dpi=110); plt.show()
print(M.astype(float).round(2).to_string())

                  realized_vol  avg_corr  pc1_share   VIX  MOVE  drawdown  var_breaches
mps_entanglement         -0.21     -0.31      -0.31 -0.19 -0.03      0.02         -0.08
mps_total_corr            0.51      0.89       0.89  0.48 -0.03      0.01          0.42
mps_mean_MI               0.51      0.90       0.90  0.48 -0.02     -0.01          0.42
mps_tail_state            0.25      0.64       0.64  0.22 -0.19     -0.13          0.17
mps_tail_asym            -0.01      0.21       0.21  0.01 -0.20     -0.04         -0.01
mps_corr_len              0.17      0.44       0.44  0.12 -0.04      0.03          0.20
mps_bond_dim             -0.21     -0.31      -0.31 -0.19 -0.03      0.02         -0.08


## 3 · Lead–lag vs VIX — does any MPS indicator move *before* VIX?

In [4]:
def leadlag(a,b,maxlag=6):
    a=a.diff().dropna(); b=b.diff().dropna(); idx=a.index.intersection(b.index); a,b=a.loc[idx],b.loc[idx]
    return pd.Series({L: a.corr(b.shift(-L)) for L in range(-maxlag,maxlag+1)})
fig, ax = plt.subplots(figsize=(9,4))
for feat in ["mps_entanglement","mps_total_corr","mps_tail_state"]:
    ll = leadlag(df[feat], df["VIX"]); ax.plot(ll.index, ll.values, marker="o", label=feat)
ax.axvline(0, color="k", lw=0.5); ax.set_xlabel("lag (months): <0 = MPS leads VIX"); ax.set_ylabel("corr of monthly Δ")
ax.set_title("Lead-lag vs VIX"); ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(P("data/rv_leadlag.png"), dpi=110); plt.show()
for feat in ["mps_entanglement","mps_total_corr","mps_tail_state","mps_tail_asym"]:
    ll = leadlag(df[feat], df["VIX"]); L=ll.abs().idxmax(); print("%-18s peak |corr| at lag %+d (corr=%.2f)  [neg=leads]" % (feat, L, ll[L]))

mps_entanglement   peak |corr| at lag -2 (corr=-0.10)  [neg=leads]
mps_total_corr     peak |corr| at lag +1 (corr=0.42)  [neg=leads]
mps_tail_state     peak |corr| at lag +1 (corr=0.13)  [neg=leads]
mps_tail_asym      peak |corr| at lag -1 (corr=-0.05)  [neg=leads]


## 4 · Redundancy — UNIQUE signal beyond vol / correlation / absorption
Regress each MPS feature on {realized_vol, avg_corr, pc1_share}. **1−R² = unique variation** the classical
trio can't explain. High 1−R² ⇒ a non-redundant indicator worth recommending.

In [5]:
def z(s): return (s-s.mean())/s.std()
Xc = np.column_stack([np.ones(len(df)), z(df["realized_vol"]), z(df["avg_corr"]), z(df["pc1_share"])])
red = []
for a in mps_cols:
    y = z(df[a]).values; beta,_,_,_ = lstsq(Xc, y, rcond=None); yhat=Xc@beta
    r2 = 1-((y-yhat)**2).sum()/((y-y.mean())**2).sum(); red.append((a, r2, 1-r2))
redf = pd.DataFrame(red, columns=["mps_feature","R2_by_classical","unique_1minusR2"]).set_index("mps_feature").sort_values("unique_1minusR2", ascending=False)
print(redf.round(3).to_string())
fig, ax = plt.subplots(figsize=(9,4)); ax.barh(redf.index, redf["unique_1minusR2"], color="#2c7fb8")
ax.set_xlabel("unique variation (1 - R^2) not explained by vol/corr/absorption"); ax.set_title("Non-redundancy of MPS indicators")
plt.tight_layout(); plt.savefig(P("data/rv_redundancy.png"), dpi=110); plt.show()

                  R2_by_classical  unique_1minusR2
mps_feature                                       
mps_tail_asym               0.073            0.927
mps_entanglement            0.111            0.889
mps_bond_dim                0.111            0.889
mps_corr_len                0.197            0.803
mps_tail_state              0.233            0.767
mps_mean_MI                 0.851            0.149
mps_total_corr              0.877            0.123


## 5 · Crisis vs calm — which indicators discriminate stress?

In [6]:
flag = pd.Series(False, index=df.index)
for s,e,_ in CRISES: flag |= (df.index>=pd.Timestamp(s)) & (df.index<=pd.Timestamp(e))
comp = pd.DataFrame({"calm_mean": df[~flag].mean(), "crisis_mean": df[flag].mean()})
comp["separation"] = (df[flag].mean()-df[~flag].mean())/df.std()
print("windows: %d crisis, %d calm" % (int(flag.sum()), int((~flag).sum())))
print(comp.loc[mps_cols+cls_cols].round(3).to_string())
fig, ax = plt.subplots(figsize=(9,5)); sep = comp.loc[mps_cols+cls_cols,"separation"].sort_values()
colors = ["#2c7fb8" if i.startswith("mps_") else "#999999" for i in sep.index]
ax.barh(sep.index, sep.values, color=colors); ax.set_xlabel("standardized crisis - calm gap (blue=MPS)")
ax.set_title("Crisis discrimination"); plt.tight_layout(); plt.savefig(P("data/rv_crisis.png"), dpi=110); plt.show()

windows: 34 crisis, 194 calm
                  calm_mean  crisis_mean  separation
mps_entanglement      2.044        2.045       0.130
mps_total_corr        9.926       10.661       0.598
mps_mean_MI           0.314        0.373       0.575
mps_tail_state        0.001        0.003       0.639
mps_tail_asym         0.037        0.016      -0.193
mps_corr_len          1.230        1.244       0.096
mps_bond_dim          7.721        7.732       0.129
realized_vol          0.309        0.365       0.254
avg_corr              0.731        0.768       0.412
pc1_share             0.755        0.789       0.414
VIX                  19.038       21.321       0.357
MOVE                 82.843       93.921       0.396
drawdown             -0.295       -0.379      -0.370
var_breaches          3.619        6.118       0.775


## 6 · Verdict — feature recommendation
*(Read against the tables/plots above.)*
- **Validation:** MPS features with high |ρ| to VIX/absorption/avg-corr (§2) confirm the MPS re-derives standard
  risk structure from the joint density — a sanity check, not a weakness.
- **Non-redundant features (§4):** those with the highest **1−R²** carry signal beyond vol/correlation/absorption
  — the genuine value-add (expect the higher-order/tail features: tail-state prob, tail-asymmetry, correlation
  length, which have no linear-risk analogue).
- **Crisis discrimination (§5):** features with the largest standardized crisis−calm gap are the best stress flags.
- **Lead-lag (§3):** report honestly whether any MPS feature *leads* VIX (negative-lag peak) or is coincident.
- **Recommendation:** keep the MPS indicators that are BOTH crisis-sensitive AND non-redundant; drop any that are
  both redundant (high R²) and crisis-insensitive.